# TP Scikit-learn · Inside Airbnb Buenos Aires
**Notebook delgado** (clase 15): la lógica vive en `src/` y está probada con pytest; acá solo se
orquesta, se verifica y se comunica.

1. Validar que los datos son los de Inside Airbnb.
2. Revisar calidad (faltantes, tipos, extremos).
3. Correr el proyecto 1 (regresión lineal) y el 4 (k-NN).

Antes de compartir: *Kernel → Restart & Run All*.

In [1]:
# Celda 1 · Importaciones: el notebook está en notebooks/, la raíz del proyecto es la carpeta de arriba
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))  # Permite «from src… import …»

import pandas as pd
from src.carga import cargar_listings, resumir_origen
from src.limpieza import preparar_datos
from src.proyecto1_capacidad import ejecutar_proyecto1
from src.proyecto4_similares import ejecutar_proyecto4

In [2]:
# Celda 2 · Carga y evidencia del origen de los datos
df = cargar_listings()
origen = resumir_origen(df)
origen

{'archivo': 'listings.csv.gz',
 'mb': 18.1,
 'sha256': 'c8db6461c06c0de3efbdda6821d6d28c9aef7dff3c6fde64ac4fd3661e15b78a',
 'coincide_sha256': True,
 'filas': 29753,
 'columnas': 90,
 'fechas_scrape': ('2026-09-25', '2026-09-27'),
 'dominio': 'com'}

In [3]:
# Celda 3 · Verificaciones mínimas (si alguna falla, el notebook se detiene acá)
assert origen["coincide_sha256"], "No es el mismo archivo del informe"
assert origen["filas"] > 20_000, "El archivo parece incompleto"
assert {"price", "accommodates", "bedrooms", "beds", "minimum_nights"} <= set(df.columns)

In [4]:
# Celda 4 · Calidad: faltantes y tipos de las columnas que usan los dos proyectos
columnas = ["price", "accommodates", "bedrooms", "beds", "minimum_nights"]
pd.DataFrame({"tipo": df[columnas].dtypes.astype(str), "faltantes": df[columnas].isna().sum()})

,tipo,faltantes
price,object,2119
accommodates,int64,0
bedrooms,float64,13
beds,float64,522
minimum_nights,float64,1


In [5]:
# Celda 5 · Distribución del precio ya limpio (muy asimétrica: media > mediana)
datos, resumen = preparar_datos(df, ["accommodates"])
print(resumen)
datos["price_num"].describe().round(0)

{'filas_iniciales': 29753, 'filas_sin_faltantes': 27634, 'filas_validas': 27634, 'filas_finales': 27086, 'precio_min_admitido': 26907.14, 'precio_max_admitido': 899463.19}


count     27086.0
mean     129895.0
std       92188.0
min       26907.0
25%       79997.0
50%      106382.0
75%      147531.0
max      897415.0
Name: price_num, dtype: float64

In [6]:
# Celda 6 · Proyecto 1: capacidad → precio con regresión lineal simple
r1 = ejecutar_proyecto1(df)
assert r1["resumen"]["pendiente"] > 0  # Más huéspedes, precio esperado mayor


PROYECTO 1 · Capacidad (accommodates) → precio · Regresión lineal simple
Modelo elegido: Regresión lineal simple: el precio es un número continuo (regresión), hay un solo predictor y se pide una regla explicable; la pendiente dice cuánto suma cada huésped.
Limpieza: {'filas_iniciales': 29753, 'filas_sin_faltantes': 27634, 'filas_validas': 27634, 'filas_finales': 27086, 'precio_min_admitido': 26907.14, 'precio_max_admitido': 899463.19}
Entrenamiento: 20314 anuncios · Prueba: 6772 anuncios
Intercepto (β0): 40,230.72 ARS
Pendiente  (β1): 30,453.79 ARS por cada huésped adicional
Modelo (test)          MAE     48,485 · RMSE     79,839 · R² 0.261
Modelo (train)         MAE     48,450 · RMSE     79,450 · R² 0.254
Base: promedio (test)  MAE     57,818 · RMSE     92,855 · R² -0.000
Estimación para 4 huéspedes: 162,046 ARS por noche

Recta vs. precio mediano observado por capacidad:
 accommodates  anuncios  mediana_real    recta
            1      1114       46836.0  70685.0
            2     1

In [7]:
# Celda 7 · Proyecto 4: alojamientos similares → precio con k-NN (k=5)
r4 = ejecutar_proyecto4(df, analizar_k=False)  # el análisis de k está en main.py
assert r4["resumen"]["metricas_test"]["RMSE"] < r4["resumen"]["metricas_base"]["RMSE"]  # Mejor que predecir el promedio


PROYECTO 4 · Alojamientos similares → precio · k-NN para regresión (k=5)
Modelo elegido: k vecinos más cercanos para regresión (KNeighborsRegressor) en un Pipeline con StandardScaler: se valora el anuncio promediando el precio de los más parecidos; el precio es numérico (Regressor) y k-NN mide distancias, por eso se escala sin fuga.
Limpieza: {'filas_iniciales': 29753, 'filas_sin_faltantes': 27143, 'filas_validas': 27143, 'filas_finales': 26599, 'precio_min_admitido': 27057.33, 'precio_max_admitido': 895060.34}
Entrenamiento: 19949 anuncios · Prueba: 6650 anuncios
Parámetros que aprendió StandardScaler (solo con train):
                media  desvio
accommodates     2.95    1.51
bedrooms         1.14    0.88
beds             1.93    1.48
minimum_nights   4.08   13.21
k-NN escalado (test)     MAE     55,569 · RMSE     85,465 · R² 0.095
k-NN escalado (train)    MAE     49,833 · RMSE     77,852 · R² 0.297
k-NN SIN escalar (test)  MAE     54,088 · RMSE     83,871 · R² 0.129
Base: promedio

## Conclusiones
La lectura de cada número, el modelo elegido y las limitaciones están en `docs/informe.md`.